# 06 — NLP Research Corpus: Semantic Search over Critical Minerals Data

This notebook builds a **semantic search engine** over the BGS critical minerals production dataset.  
Each `(commodity, country)` pair is converted into a short natural-language document, embedded with a  
pre-trained sentence transformer, indexed with FAISS, and then made searchable via free-text queries.

**Pipeline at a glance**
1. Load & aggregate the BGS production CSV into one document per `(commodity, country)` pair.
2. Encode all documents with **all-MiniLM-L6-v2** (384-dimensional, ~80 MB, fast CPU inference).
3. L2-normalise embeddings and load into a **FAISS IndexFlatIP** (inner product ≡ cosine similarity).
4. Run free-text queries — the model encodes the query on the fly and retrieves the nearest neighbours.
5. Visualise the full embedding space in 2D with **UMAP** and **Plotly**.

## 1. Setup — install dependencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import faiss
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
DATA_DIR = (NOTEBOOK_DIR / "../data/bgs_data").resolve()
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

print(f"Data directory : {DATA_DIR}")
print(f"CSV exists     : {CSV_PATH.exists()}")

## 2. Load the BGS production data

In [ ]:
raw = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Rows: {len(raw):,}  |  Columns: {raw.shape[1]}")
print("\nColumn names:")
print(raw.columns.tolist())
raw.head(3)

## 3. Build the NLP corpus

For each `(commodity, country)` pair we construct a short, information-dense sentence.  
The template is:

> *{country} produces {commodity}. Production history spans {min_year}–{max_year} ({N} years of data).  
> Latest production ({latest_year}): {qty} {units}. Peak production: {peak_qty} {units} in {peak_year}.  
> Average production: {avg_qty} {units}.*

Any non-null notes from the source data are appended (truncated to 200 characters).

Keeping the text factual and structured gives the sentence transformer enough signal to distinguish  
minerals, geographies, and orders of magnitude — all without any fine-tuning.

In [ ]:
def _fmt(value, decimals: int = 0) -> str:
    """Format a numeric value with commas; fall back to 'N/A' for NaN."""
    if pd.isna(value):
        return "N/A"
    if decimals == 0:
        return f"{value:,.0f}"
    return f"{value:,.{decimals}f}"


def build_document(group: pd.DataFrame) -> dict:
    """Given a subset of rows for one (commodity, country) pair, return a corpus record."""
    row0 = group.iloc[0]
    commodity = row0["commodity"]
    country   = row0["country"]
    iso3      = row0.get("country_iso3", "")
    units     = row0["units"] if pd.notna(row0.get("units")) else "units"

    valid = group.dropna(subset=["quantity"])

    if valid.empty:
        return None

    min_year    = int(valid["year"].min())
    max_year    = int(valid["year"].max())
    n_years     = valid["year"].nunique()

    latest_row  = valid.sort_values("year").iloc[-1]
    latest_year = int(latest_row["year"])
    latest_qty  = latest_row["quantity"]

    peak_idx    = valid["quantity"].idxmax()
    peak_row    = valid.loc[peak_idx]
    peak_qty    = peak_row["quantity"]
    peak_year   = int(peak_row["year"])

    avg_qty     = valid["quantity"].mean()

    text = (
        f"{country} produces {commodity}. "
        f"Production history spans {min_year}-{max_year} ({n_years} years of data). "
        f"Latest production ({latest_year}): {_fmt(latest_qty)} {units}. "
        f"Peak production: {_fmt(peak_qty)} {units} in {peak_year}. "
        f"Average production: {_fmt(avg_qty)} {units}."
    )

    # Append notes if available
    notes_parts = []
    for col in ("table_notes", "figure_notes"):
        note = row0.get(col, None)
        if pd.notna(note) and str(note).strip():
            notes_parts.append(str(note).strip())
    if notes_parts:
        notes_combined = " ".join(notes_parts)[:200]
        text += f" Notes: {notes_combined}"

    return {
        "commodity"         : commodity,
        "country"           : country,
        "iso3"              : iso3,
        "latest_production" : latest_qty,
        "text"              : text,
    }


# Build corpus — one record per (commodity, country)
records = []
for (commodity, country), grp in raw.groupby(["commodity", "country"], sort=False):
    doc = build_document(grp)
    if doc is not None:
        records.append(doc)

corpus_df = pd.DataFrame(records).reset_index(drop=True)

print(f"Corpus size: {len(corpus_df):,} documents")
print(f"Unique commodities : {corpus_df['commodity'].nunique()}")
print(f"Unique countries   : {corpus_df['country'].nunique()}")
corpus_df.head(5)

In [ ]:
# Preview a sample document
sample_idx = corpus_df[corpus_df["commodity"].str.contains("lithium", case=False)].index[0]
print(corpus_df.loc[sample_idx, "text"])

## 4. Generate sentence embeddings

**Model:** `all-MiniLM-L6-v2`  
- Architecture: MiniLM (6-layer transformer), fine-tuned on 1 billion sentence pairs for semantic similarity.  
- Output: 384-dimensional dense vectors — compact, fast, and highly expressive for short texts.  
- Size: ~80 MB — runs comfortably on CPU.

After encoding we **L2-normalise** every vector so that inner product equals cosine similarity.  
This lets us use the efficient `IndexFlatIP` in FAISS.

In [ ]:
print("Loading sentence transformer model: all-MiniLM-L6-v2 ...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

texts = corpus_df["text"].tolist()

print(f"\nEncoding {len(texts):,} documents (batch_size=64) ...")
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True,
)

# L2 normalise so inner product == cosine similarity
faiss.normalize_L2(embeddings)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"dtype: {embeddings.dtype}")

## 5. Build FAISS index & semantic search

`IndexFlatIP` performs exact nearest-neighbour search using the inner product.  
Because embeddings are L2-normalised, this is equivalent to cosine similarity — no approximation,  
no quantisation.  At corpus sizes up to ~1 M documents this remains fast on CPU.

In [ ]:
DIM = embeddings.shape[1]          # 384

index = faiss.IndexFlatIP(DIM)
index.add(embeddings)

print(f"FAISS index type : IndexFlatIP")
print(f"Vector dimension : {DIM}")
print(f"Vectors indexed  : {index.ntotal:,}")

In [ ]:
def semantic_search(query: str, top_k: int = 10) -> pd.DataFrame:
    """
    Search the FAISS index for documents most similar to *query*.

    Parameters
    ----------
    query  : Free-text search string.
    top_k  : Number of results to return.

    Returns
    -------
    pd.DataFrame with columns [score, commodity, country, text_snippet].
    """
    q_vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)

    scores, indices = index.search(q_vec, top_k)

    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        rec = corpus_df.iloc[idx]
        rows.append({
            "score"        : round(float(score), 4),
            "commodity"    : rec["commodity"],
            "country"      : rec["country"],
            "text_snippet" : rec["text"][:160] + "…",
        })

    return pd.DataFrame(rows)

### 5.1 Demo queries

In [ ]:
QUERIES = [
    "Which countries produce lithium for batteries?",
    "Rare earth element production in Africa",
    "Cobalt mining in Democratic Republic of Congo",
    "Platinum group metals supply chain",
]

for q in QUERIES:
    print(f"\n{'='*70}")
    print(f"Query: {q}")
    print('='*70)
    results = semantic_search(q, top_k=10)
    print(results.to_string(index=False))

## 6. Visualise the embedding space with UMAP

**UMAP** (Uniform Manifold Approximation and Projection) reduces the 384-dimensional embeddings to 2D  
while preserving local neighbourhood structure.  Clusters in this projection correspond to groups of  
`(commodity, country)` pairs that are semantically similar — e.g. all platinum-group-metal producers  
should cluster together, as should all African rare-earth producers.

Parameters used:
- `n_neighbors=15` — balance between local and global structure.
- `min_dist=0.1` — how tightly packed clusters are.
- `metric="cosine"` — consistent with the FAISS index similarity measure.

In [ ]:
import umap

print("Running UMAP dimensionality reduction (384D -> 2D) ...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
    n_components=2,
    verbose=False,
)
umap_coords = reducer.fit_transform(embeddings)

plot_df = corpus_df.copy()
plot_df["umap_x"] = umap_coords[:, 0]
plot_df["umap_y"] = umap_coords[:, 1]

print(f"UMAP output shape: {umap_coords.shape}")
plot_df[["commodity", "country", "umap_x", "umap_y"]].head(5)

In [ ]:
# Limit legend to top-N commodities by document count for readability
TOP_N = 20
top_commodities = (
    plot_df["commodity"].value_counts().head(TOP_N).index.tolist()
)
plot_df["commodity_label"] = plot_df["commodity"].where(
    plot_df["commodity"].isin(top_commodities), other="other"
)

fig = px.scatter(
    plot_df,
    x="umap_x",
    y="umap_y",
    color="commodity_label",
    hover_name="country",
    hover_data={
        "commodity"         : True,
        "latest_production" : ":,.0f",
        "umap_x"            : False,
        "umap_y"            : False,
        "commodity_label"   : False,
    },
    title="UMAP Projection of Critical Minerals Corpus (all-MiniLM-L6-v2, 384D → 2D)",
    labels={
        "umap_x"          : "UMAP 1",
        "umap_y"          : "UMAP 2",
        "commodity_label" : "Commodity",
    },
    opacity=0.75,
    width=1000,
    height=700,
)

fig.update_traces(marker=dict(size=6))
fig.update_layout(
    legend=dict(title="Commodity (top 20)", itemsizing="constant"),
    font=dict(family="Arial", size=12),
)

fig.show()

## 7. Summary

| Step | Detail |
|------|--------|
| **Model** | `all-MiniLM-L6-v2` — 384-dim, ~80 MB, fine-tuned for semantic similarity |
| **Corpus** | One document per `(commodity, country)` pair from BGS production data |
| **Index** | FAISS `IndexFlatIP` — exact cosine search over L2-normalised vectors |
| **Visualisation** | UMAP 2D projection, coloured by commodity, hover shows country |

**Next steps**
- Fine-tune the model on domain-specific critical-minerals text (journal abstracts, USGS reports).
- Swap `IndexFlatIP` for `IndexIVFFlat` or `IndexHNSWFlat` for approximate search at larger scales.
- Build an interactive Streamlit/Dash app that wraps `semantic_search()`.
- Augment corpus documents with trade-flow data, price series, or supply-chain risk scores.